# 07 · Comparaison de modèles pour le risque de crédit

**Contexte** : le service crédit à la consommation d'une banque fictive note les demandes de prêt avec une régression logistique. La direction des risques se demande si une forêt aléatoire ou un gradient boosting détecterait mieux les défauts, et si le gain justifie la perte d'explicabilité (il faut pouvoir justifier un refus de crédit).

**Règle du jeu** : tous les choix (variables, réglages, seuil) sont faits par **validation croisée sur le train**. Le jeu de test n'est ouvert qu'**une seule fois**, à l'étape 9.

> Données entièrement fictives : 12 070 demandes de prêt générées par `generer_donnees.py`.

## 1. Chargement et audit

In [1]:
import pandas as pd
import numpy as np

In [2]:
credit = pd.read_csv("demandes_credit.csv", sep=";", decimal=",")
credit.shape

(12070, 19)

In [3]:
credit.head()

,ID_Demande,Date_Demande,Age,Situation_Pro,Anciennete_Emploi_Ans,Statut_Logement,Nb_Enfants,Revenu_Mensuel_EUR,Epargne_EUR,Objet_Pret,Montant_EUR,Duree_Mois,Taux_Annuel_Pct,Nb_Credits_En_Cours,Charges_Credits_Mensuelles_EUR,Nb_Incidents_24M,Jours_Decouvert_12M,Nb_Relances_Recouvrement,Defaut
0,D100395,23/04/2024,38,CDD,2.4,Propriétaire,0,1650.0,570.0,Rachat de crédits,2700.0,48,4.90,3,442.0,1,2,0,Non
1,D106302,26/11/2025,54,CDI,2.5,Locataire,2,3060.0,8050.0,Travaux,8500.0,36,3.70,0,0.0,0,44,0,Non
2,D110810,03/07/2023,35,CDI,3.1,Locataire,3,2980.0,NaN,Consommation,8700.0,24,3.65,0,0.0,0,8,5,Oui
3,D109380,22/09/2024,39,Indépendant,9.9,Propriétaire,2,4600.0,5750.0,Études,10300.0,72,4.52,1,176.0,0,12,0,Non
4,D110187,16/08/2024,69,Retraité,NaN,Locataire,2,2160.0,1410.0,Consommation,11500.0,72,4.74,0,0.0,0,12,0,Non


In [4]:
credit.describe()

,Age,Anciennete_Emploi_Ans,Nb_Enfants,Revenu_Mensuel_EUR,Epargne_EUR,Montant_EUR,Duree_Mois,Taux_Annuel_Pct,Nb_Credits_En_Cours,Charges_Credits_Mensuelles_EUR,Nb_Incidents_24M,Jours_Decouvert_12M,Nb_Relances_Recouvrement
count,12070.000000,9973.000000,12070.000000,11880.000000,1.132000e+04,12070.000000,12070.000000,12070.000000,12070.000000,12070.000000,12070.000000,12070.000000,12070.000000
mean,45.649959,5.487607,1.129660,2626.739057,8.930293e+03,15507.729909,47.404474,4.378548,1.230075,250.941259,0.250373,15.032643,0.469594
std,24.458616,5.246274,1.141061,1097.907417,2.147206e+04,10640.864926,19.506598,0.695607,1.266799,285.965155,0.501334,17.115215,1.169897
min,0.000000,0.000000,0.000000,-5750.000000,3.000000e+01,1100.000000,0.000000,1.830000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,35.000000,1.600000,0.000000,1870.000000,1.650000e+03,8400.000000,36.000000,3.900000,0.000000,0.000000,0.000000,4.000000,0.000000
50%,43.000000,3.900000,1.000000,2480.000000,3.860000e+03,12700.000000,48.000000,4.370000,1.000000,191.000000,0.000000,10.000000,0.000000
75%,53.000000,7.700000,2.000000,3230.000000,9.262500e+03,19300.000000,60.000000,4.850000,2.000000,385.000000,0.000000,20.000000,0.000000
max,999.000000,43.700000,4.000000,9080.000000,1.053910e+06,120000.000000,84.000000,7.220000,6.000000,1910.000000,4.000000,260.000000,11.000000


Premières anomalies visibles dans le `describe()` :
- `Age` va de 0 à 999 ans : impossible, un emprunteur a entre 18 et 99 ans ;
- `Revenu_Mensuel_EUR` descend à -5 750 € : un revenu négatif n'existe pas ;
- `Duree_Mois` a un minimum de 0 : un prêt sur 0 mois n'a pas de sens (et ferait planter la formule de mensualité).

In [5]:
credit.isna().sum()

ID_Demande                           0
Date_Demande                         0
Age                                  0
Situation_Pro                        0
Anciennete_Emploi_Ans             2097
Statut_Logement                    250
Nb_Enfants                           0
Revenu_Mensuel_EUR                 190
Epargne_EUR                        750
Objet_Pret                           0
Montant_EUR                          0
Duree_Mois                           0
Taux_Annuel_Pct                      0
Nb_Credits_En_Cours                  0
Charges_Credits_Mensuelles_EUR       0
Nb_Incidents_24M                     0
Jours_Decouvert_12M                  0
Nb_Relances_Recouvrement             0
Defaut                               0
dtype: int64

In [6]:
credit.nunique()

ID_Demande                        12000
Date_Demande                       1095
Age                                  70
Situation_Pro                         9
Anciennete_Emploi_Ans               279
Statut_Logement                       4
Nb_Enfants                            5
Revenu_Mensuel_EUR                  647
Epargne_EUR                        2701
Objet_Pret                            5
Montant_EUR                         576
Duree_Mois                            8
Taux_Annuel_Pct                     421
Nb_Credits_En_Cours                   7
Charges_Credits_Mensuelles_EUR     1063
Nb_Incidents_24M                      5
Jours_Decouvert_12M                 136
Nb_Relances_Recouvrement             12
Defaut                                2
dtype: int64

### Doublons et variable cible

In [7]:
credit = credit.drop_duplicates()
credit.shape

(12000, 19)

In [8]:
credit["Defaut"] = credit["Defaut"].map({"Oui": 1, "Non": 0})
print(credit["Defaut"].value_counts(dropna=False))
print(credit["Defaut"].value_counts(normalize=True))

Defaut
0    10730
1     1270
Name: count, dtype: int64
Defaut
0    0.894167
1    0.105833
Name: proportion, dtype: float64


70 lignes dupliquées supprimées. Il reste 12 000 demandes, dont **10,6 % de défauts** : les classes sont déséquilibrées, l'accuracy ne sera donc pas un bon indicateur.

### Valeurs manquantes : erreur ou information ?

In [9]:
pd.crosstab(credit["Situation_Pro"], credit["Anciennete_Emploi_Ans"].isna())

Anciennete_Emploi_Ans,False,True
Situation_Pro,,
C.D.I.,46,0
CDD,1565,0
CDI,5321,0
CDI,64,0
Fonctionnaire,1680,0
Indépendant,1197,0
Retraité,0,1583
Sans emploi,0,504
cdi,40,0


Les anciennetés manquantes concernent **uniquement** les retraités et les sans-emploi : ce n'est pas une erreur de saisie, c'est une information (pas d'emploi actuel). On garde la colonne et on laissera `SimpleImputer(add_indicator=True)` créer une colonne « ancienneté manquante ».

Le tableau révèle aussi un problème de libellés : `CDI` apparaît sous quatre formes (`CDI`, `CDI ` avec un espace, `cdi`, `C.D.I.`).

### Libellés incohérents

In [10]:
credit["Situation_Pro"] = credit["Situation_Pro"].str.strip().replace({"cdi": "CDI", "C.D.I.": "CDI"})
credit["Situation_Pro"].value_counts()

Situation_Pro
CDI              5471
Fonctionnaire    1680
Retraité         1583
CDD              1565
Indépendant      1197
Sans emploi       504
Name: count, dtype: int64

In [11]:
credit["Statut_Logement"].value_counts(dropna=False)

Statut_Logement
Propriétaire    5537
Locataire       4690
Hébergé         1444
NaN              249
locataire         80
Name: count, dtype: int64

In [12]:
# "locataire" en minuscules -> "Locataire", vides -> "Inconnu"
credit["Statut_Logement"] = credit["Statut_Logement"].str.strip().str.capitalize().fillna("Inconnu")
credit["Statut_Logement"].value_counts(dropna=False)

Statut_Logement
Propriétaire    5537
Locataire       4770
Hébergé         1444
Inconnu          249
Name: count, dtype: int64

### Valeurs impossibles

In [13]:
# Valeurs impossibles -> NaN (le SimpleImputer les complétera dans le pipeline, avec la médiane du train)
credit.loc[(credit["Age"] < 18) | (credit["Age"] > 99), "Age"] = np.nan
credit.loc[credit["Revenu_Mensuel_EUR"] <= 0, "Revenu_Mensuel_EUR"] = np.nan
credit.loc[credit["Duree_Mois"] <= 0, "Duree_Mois"] = np.nan

print("Âges impossibles :", credit["Age"].isna().sum())
print("Revenus manquants ou impossibles :", credit["Revenu_Mensuel_EUR"].isna().sum())
print("Durées impossibles :", credit["Duree_Mois"].isna().sum())

Âges impossibles : 12
Revenus manquants ou impossibles : 201
Durées impossibles : 8


On remplace par `NaN` plutôt que de supprimer les lignes : en production, la banque doit quand même noter la demande d'un client dont l'âge a été mal saisi. Pour le revenu et l'épargne manquants, on ne met pas 0 : « inconnu » n'est pas « zéro » (un revenu à 0 ferait croire que le client n'a aucun revenu, et diviserait par zéro dans le taux d'endettement).

## 2. Une colonne de trop : `Nb_Relances_Recouvrement`

Les relances de recouvrement sont envoyées **après** un impayé. Au moment où le conseiller étudie la demande, cette information n'existe pas encore. Si on la donne au modèle, il « lit la réponse » : c'est une **fuite de données**. Elle est testée à l'étape 4, puis exclue des variables.

`ID_Demande` (identifiant) et `Date_Demande` n'ont pas non plus de sens prédictif ; elles ne seront pas données au modèle.

## 3. Variables métier : mensualité et taux d'endettement

In [14]:
# Mensualité du nouveau prêt (formule d'annuité)
t = credit["Taux_Annuel_Pct"] / 100 / 12
credit["Mensualite_EUR"] = credit["Montant_EUR"] * t / (1 - (1 + t) ** (-credit["Duree_Mois"]))

# Taux d'endettement après le prêt
credit["Taux_Endettement"] = (
    (credit["Mensualite_EUR"] + credit["Charges_Credits_Mensuelles_EUR"]) / credit["Revenu_Mensuel_EUR"]
)

# Dépassement du plafond français de 35 % (reste vide si le taux est inconnu)
credit["Depasse_35"] = (credit["Taux_Endettement"] > 0.35).astype(int)
credit.loc[credit["Taux_Endettement"].isna(), "Depasse_35"] = np.nan

credit[["Mensualite_EUR", "Taux_Endettement"]].describe()

,Mensualite_EUR,Taux_Endettement
count,11992.000000,11791.000000
mean,446.134454,0.318766
std,436.396672,0.319062
min,24.396809,0.005224
25%,192.764976,0.126815
50%,315.437614,0.232324
75%,539.809694,0.402301
max,8928.140388,5.085223


In [15]:
print(credit["Depasse_35"].value_counts(normalize=True, dropna=False))
credit.groupby("Depasse_35")["Defaut"].agg(["count", "mean"])

Depasse_35
0.0    0.683917
1.0    0.298667
NaN    0.017417
Name: proportion, dtype: float64


,count,mean
Depasse_35,,
0.0,8207,0.041550
1.0,3584,0.251116


**30 % des demandes dépassent le plafond de 35 %**, et leur taux de défaut est de **25 %**, contre **4 %** en dessous : le risque est multiplié par 6 au-delà du plafond. C'est un effet de **seuil**, pas une ligne droite.

## 4. Découpage train / test et prétraitement

In [16]:
from sklearn.model_selection import train_test_split

x = credit.drop("Defaut", axis=1)
y = credit["Defaut"]

X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train :", X_train.shape, "| taux de défaut :", round(y_train.mean(), 3))
print("Test  :", X_test.shape, "| taux de défaut :", round(y_test.mean(), 3))

Train : (9600, 21) | taux de défaut : 0.106
Test  : (2400, 21) | taux de défaut : 0.106


**À partir d'ici, `X_test` reste au coffre jusqu'à l'étape 9.**

In [17]:
numeric_features_base = [
    "Age",
    "Anciennete_Emploi_Ans",
    "Nb_Enfants",
    "Revenu_Mensuel_EUR",
    "Epargne_EUR",
    "Montant_EUR",
    "Duree_Mois",
    "Taux_Annuel_Pct",
    "Nb_Credits_En_Cours",
    "Charges_Credits_Mensuelles_EUR",
    "Nb_Incidents_24M",
    "Jours_Decouvert_12M",
    "Mensualite_EUR",
    "Taux_Endettement"
]
numeric_features = numeric_features_base + ["Depasse_35"]

categorical_features = [
    "Situation_Pro",
    "Statut_Logement",
    "Objet_Pret"
]

Une seule fonction construit le prétraitement de tous les modèles :
- variables numériques : médiane + indicateur de valeur manquante, puis standardisation (utile pour la logistique, inutile pour les arbres) ;
- variables catégorielles : valeur la plus fréquente + one-hot.

Le `ColumnTransformer` n'utilise que les colonnes listées : `ID_Demande`, `Date_Demande` et `Nb_Relances_Recouvrement` sont ignorées tant qu'on ne les ajoute pas.

In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def pretraitement(colonnes_num, standardiser=True):
    etapes_num = [("imputer", SimpleImputer(strategy="median", add_indicator=True))]
    if standardiser:
        etapes_num.append(("scaler", StandardScaler()))

    categoric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline(steps=etapes_num), colonnes_num),
            ("cat", categoric_transformer, categorical_features)
        ]
    )

In [19]:
from sklearn.linear_model import LogisticRegression

def logistique(colonnes_num, poids="balanced"):
    return Pipeline(
        steps=[
            ("preprocessor", pretraitement(colonnes_num)),
            ("classifier", LogisticRegression(max_iter=1000, class_weight=poids))
        ]
    )

### Validation croisée

Plutôt qu'un seul score sur un seul découpage, chaque modèle est évalué 5 fois : le train est coupé en 5 paquets, le modèle apprend sur 4 et il est noté sur le 5ᵉ, en changeant de paquet à chaque tour. On obtient une **moyenne** (score fiable) et un **écart-type** (stabilité). Tous les modèles sont évalués sur **les mêmes 5 plis**.

In [20]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate, cross_val_predict

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluer(nom, modele):
    res = cross_validate(modele, X_train, y_train, cv=cv, scoring="roc_auc", return_train_score=True)
    print(f"{nom:32} | AUC apprentissage : {res['train_score'].mean():.3f} | "
          f"AUC validation : {res['test_score'].mean():.3f} ± {res['test_score'].std():.3f}")
    return res["test_score"]

### Test de la fuite de données

In [21]:
evaluer("Logistique AVEC relances", logistique(numeric_features + ["Nb_Relances_Recouvrement"]))
evaluer("Logistique SANS relances", logistique(numeric_features));

Logistique AVEC relances         | AUC apprentissage : 0.984 | AUC validation : 0.981 ± 0.007


Logistique SANS relances         | AUC apprentissage : 0.799 | AUC validation : 0.791 ± 0.006


Avec `Nb_Relances_Recouvrement`, l'AUC passe d'environ 0,79 à **0,98**. Ce gain est suspect : les relances sont envoyées après un impayé, donc cette colonne contient en partie la réponse. Un modèle qui s'appuie dessus serait excellent dans le notebook et inutile en production (pour un nouveau client, elle vaut toujours 0). **La colonne est exclue pour la suite.**

> Rappel : l'AUC se lit par paires. Si on prend au hasard un dossier en défaut et un dossier sans défaut, c'est la probabilité que le modèle donne la note de risque la plus élevée au dossier en défaut (50 % = hasard, 100 % = parfait).

## 5. Référence : la régression logistique

In [22]:
scores_logit_base = evaluer("Logistique sans Depasse_35", logistique(numeric_features_base))
scores_logit = evaluer("Logistique avec Depasse_35", logistique(numeric_features))
print("\nAUC par pli (avec Depasse_35) :", scores_logit.round(3))

Logistique sans Depasse_35       | AUC apprentissage : 0.766 | AUC validation : 0.756 ± 0.013


Logistique avec Depasse_35       | AUC apprentissage : 0.799 | AUC validation : 0.791 ± 0.006

AUC par pli (avec Depasse_35) : [0.795 0.788 0.799 0.781 0.791]


- Les 5 plis vont de 0,781 à 0,799 : un score calculé sur un seul découpage peut varier de près de 2 points selon le hasard. D'où l'intérêt de la validation croisée pour comparer des modèles.
- `Depasse_35` fait gagner **3,5 points** d'AUC à la logistique (0,756 → 0,791) : une logistique ne trace que des effets en ligne droite, elle ne peut pas représenter seule le saut de risque à 35 %. La variable métier le lui donne.

**Barre à battre : AUC = 0,791 ± 0,006.**

## 6. Forêt aléatoire

Un arbre pose des questions du type « revenu < 2 000 € ? » : la standardisation est inutile, on garde seulement l'imputation et le one-hot.

In [23]:
from sklearn.ensemble import RandomForestClassifier

def foret(colonnes_num, feuille_min):
    return Pipeline(
        steps=[
            ("preprocessor", pretraitement(colonnes_num, standardiser=False)),
            ("classifier", RandomForestClassifier(
                n_estimators=300,
                min_samples_leaf=feuille_min,
                class_weight="balanced",
                n_jobs=-1,
                random_state=42
            ))
        ]
    )

evaluer("Forêt, feuille de 1", foret(numeric_features, 1))
evaluer("Forêt, feuille de 20, sans Depasse_35", foret(numeric_features_base, 20))
scores_foret = evaluer("Forêt, feuille de 20", foret(numeric_features, 20))

Forêt, feuille de 1              | AUC apprentissage : 1.000 | AUC validation : 0.790 ± 0.010


Forêt, feuille de 20, sans Depasse_35 | AUC apprentissage : 0.915 | AUC validation : 0.793 ± 0.009


Forêt, feuille de 20             | AUC apprentissage : 0.910 | AUC validation : 0.795 ± 0.007


- **Surapprentissage** : avec des feuilles d'un seul client, la forêt apprend le train par cœur (AUC d'apprentissage = 1,000) et retombe à 0,790 sur des dossiers nouveaux. Avec 20 clients minimum par feuille, l'écart se réduit. C'est toujours le score de **validation** qui compte.
- `Depasse_35` n'apporte presque rien à la forêt (+0,002) : un arbre trouve seul le seuil de 35 % à partir de `Taux_Endettement`.
- Forêt (0,795) et logistique avec `Depasse_35` (0,791) sont **à égalité** : l'écart est plus petit que l'écart-type. La forêt ne battait la logistique que parce qu'elle trouvait seule ce seuil.

## 7. Gradient boosting

Une forêt construit des arbres **indépendants** et fait la moyenne de leurs votes. Le boosting construit des arbres **l'un après l'autre**, chacun corrigeant les erreurs des précédents.

In [24]:
from sklearn.ensemble import HistGradientBoostingClassifier

boosting = Pipeline(
    steps=[
        ("preprocessor", pretraitement(numeric_features, standardiser=False)),
        ("classifier", HistGradientBoostingClassifier(random_state=42))
    ]
)

evaluer("Boosting par défaut", boosting);

Boosting par défaut              | AUC apprentissage : 0.992 | AUC validation : 0.785 ± 0.012


Avec ses réglages par défaut, le boosting surapprend fortement (0,992 en apprentissage contre 0,785 en validation) et fait moins bien que la forêt. On cherche de meilleurs réglages avec `GridSearchCV`, toujours sur le train.

In [25]:
from sklearn.model_selection import GridSearchCV

grille = {
    "classifier__learning_rate": [0.03, 0.1],
    "classifier__max_depth": [3, 5],
    "classifier__min_samples_leaf": [20, 100],
    "classifier__max_iter": [100, 300]
}

recherche = GridSearchCV(
    boosting,
    param_grid=grille,
    cv=cv,
    scoring="roc_auc",
    return_train_score=True,
    n_jobs=-1
)
recherche.fit(X_train, y_train)

print("Meilleurs réglages :", recherche.best_params_)
print(f"Meilleur AUC validation : {recherche.best_score_:.3f} ± {recherche.cv_results_['std_test_score'][recherche.best_index_]:.3f}")

Meilleurs réglages : {'classifier__learning_rate': 0.03, 'classifier__max_depth': 3, 'classifier__max_iter': 100, 'classifier__min_samples_leaf': 20}
Meilleur AUC validation : 0.807 ± 0.007


In [26]:
resultats = pd.DataFrame(recherche.cv_results_)
resultats[[
    "param_classifier__learning_rate",
    "param_classifier__max_depth",
    "param_classifier__min_samples_leaf",
    "param_classifier__max_iter",
    "mean_train_score",
    "mean_test_score",
    "std_test_score"
]].sort_values("mean_test_score", ascending=False).head(8).round(3)

,param_classifier__learning_rate,param_classifier__max_depth,param_classifier__min_samples_leaf,param_classifier__max_iter,mean_train_score,mean_test_score,std_test_score
0,0.03,3,20,100,0.836,0.807,0.007
1,0.03,3,100,100,0.830,0.805,0.006
2,0.03,3,20,300,0.869,0.805,0.007
3,0.03,3,100,300,0.861,0.804,0.006
9,0.10,3,100,100,0.865,0.804,0.006
8,0.10,3,20,100,0.876,0.802,0.008
5,0.03,5,100,100,0.861,0.801,0.007
7,0.03,5,100,300,0.912,0.798,0.009


- Une fois réglé, le boosting surapprend beaucoup moins (0,836 en apprentissage contre 0,807 en validation, 3 points d'écart au lieu de 21).
- Il devient le meilleur modèle : **0,807**, contre 0,795 pour la forêt (à la limite de l'écart-type) et 0,791 pour la logistique (environ 2,5 écarts-types).
- Les 5 meilleures combinaisons sont toutes entre 0,804 et 0,807 : le résultat ne dépend pas d'un réglage chanceux. Le gagnant est le réglage le plus simple de la grille (apprentissage lent, arbres peu profonds).

## 8. Choix du seuil selon les coûts, sans regarder le test

**Hypothèses de coût** : un défaut accordé coûte en moyenne **4 000 €**, un bon client refusé **600 €** de marge perdue.

`cross_val_predict` donne une probabilité pour chaque dossier du train, calculée par un modèle qui ne l'a pas vu pendant son apprentissage. On choisit le seuil sur ces probabilités : le test n'est pas utilisé.

In [27]:
COUT_DEFAUT_ACCORDE = 4000   # prêt accepté qui fait défaut
COUT_BON_REFUSE = 600        # bon client refusé (marge perdue)

def seuil_cout_optimal(probas, y_vrai):
    lignes = []
    for seuil in np.arange(0.02, 0.96, 0.01):
        refus = probas >= seuil
        defauts_accordes = ((~refus) & (y_vrai == 1)).sum()
        bons_refuses = (refus & (y_vrai == 0)).sum()
        lignes.append({
            "seuil": round(seuil, 2),
            "refus": refus.sum(),
            "defauts_accordes": defauts_accordes,
            "bons_refuses": bons_refuses,
            "cout": defauts_accordes * COUT_DEFAUT_ACCORDE + bons_refuses * COUT_BON_REFUSE
        })
    tableau = pd.DataFrame(lignes)
    return tableau.loc[tableau["cout"].idxmin()]

In [28]:
modeles = {
    "Logistique": logistique(numeric_features),
    "Forêt": foret(numeric_features, 20),
    "Boosting réglé": recherche.best_estimator_
}

print("Coût si on accepte tout le monde :", f"{y_train.sum() * COUT_DEFAUT_ACCORDE:,.0f} €\n")

probas_cv = {}
seuils_choisis = {}
for nom, modele in modeles.items():
    probas_cv[nom] = cross_val_predict(modele, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
    meilleur = seuil_cout_optimal(probas_cv[nom], y_train)
    seuils_choisis[nom] = meilleur["seuil"]
    print(f"{nom:15} | seuil : {meilleur['seuil']:.2f} | refus : {meilleur['refus']:.0f} | "
          f"défauts accordés : {meilleur['defauts_accordes']:.0f} | bons refusés : {meilleur['bons_refuses']:.0f} | "
          f"coût : {meilleur['cout']:,.0f} €")

Coût si on accepte tout le monde : 4,064,000 €



Logistique      | seuil : 0.51 | refus : 2995 | défauts accordés : 261 | bons refusés : 2240 | coût : 2,388,000 €


Forêt           | seuil : 0.56 | refus : 2237 | défauts accordés : 355 | bons refusés : 1576 | coût : 2,365,600 €


Boosting réglé  | seuil : 0.15 | refus : 2011 | défauts accordés : 378 | bons refusés : 1373 | coût : 2,335,800 €


- Le boosting est le moins coûteux : environ **52 000 € de moins** que la logistique sur 9 600 dossiers (environ 5,40 € par dossier). C'est 3 % de l'économie déjà réalisée par la logistique par rapport à « accepter tout le monde ».
- Les seuils sont très différents (0,51 et 0,56 contre 0,15) : la logistique et la forêt ont `class_weight="balanced"`, qui gonfle leurs probabilités. Le boosting n'est pas pondéré, ses probabilités sont réalistes. Son seuil est d'ailleurs proche de la valeur théorique : refuser si `p × 4 000 > (1 − p) × 600`, soit `p > 0,13`.

### La pondération change-t-elle quelque chose ?

In [29]:
logit_sans_poids = logistique(numeric_features, poids=None)
evaluer("Logistique sans pondération", logit_sans_poids)
seuil_cout_optimal(cross_val_predict(logit_sans_poids, X_train, y_train, cv=cv, method="predict_proba")[:, 1], y_train)

Logistique sans pondération      | AUC apprentissage : 0.799 | AUC validation : 0.792 ± 0.006


seuil                     0.11
refus                  2984.00
defauts_accordes        264.00
bons_refuses           2232.00
cout                2395200.00
Name: 9, dtype: float64

Même AUC, même nombre de refus et de défauts accordés, même coût à quelques milliers d'euros près : la pondération **déplace le seuil** (0,51 → 0,11) sans améliorer le modèle. L'avantage du boosting vient donc bien de son **classement** des dossiers.

### Comparer à volume de refus égal

Pour comparer deux modèles, on les place au même nombre de refus, pas au même seuil (le seuil dépend de l'échelle des probabilités).

In [30]:
for nb_refus in [2995, 2011]:   # volumes de refus optimaux de la logistique et du boosting
    print(f"--- {nb_refus} refus ---")
    for nom in ["Logistique", "Boosting réglé"]:
        probas = probas_cv[nom]
        seuil = np.sort(probas)[-nb_refus]          # seuil qui refuse exactement les nb_refus dossiers les plus risqués
        refus = probas >= seuil
        defauts_accordes = ((~refus) & (y_train == 1)).sum()
        print(f"{nom:15} | seuil : {seuil:.3f} | défauts accordés : {defauts_accordes}")

--- 2995 refus ---
Logistique      | seuil : 0.510 | défauts accordés : 261
Boosting réglé  | seuil : 0.116 | défauts accordés : 263
--- 2011 refus ---
Logistique      | seuil : 0.653 | défauts accordés : 421
Boosting réglé  | seuil : 0.150 | défauts accordés : 378


- **À 2 995 refus (31 % des demandes), les deux modèles sont à égalité** : ils repèrent aussi bien les dossiers clairement risqués.
- **À 2 011 refus, la logistique laisse passer 43 défauts de plus** (environ 200 000 € de plus) : le boosting trie mieux la « zone grise ».

Ce que ça veut dire pour la banque : si elle accepte de refuser environ 30 % des demandes, la logistique suffit. Si elle veut accepter plus de clients, le boosting lui permet d'en accepter environ 1 000 de plus sur 9 600 en gardant le risque maîtrisé.

## 9. Ouverture du coffre : évaluation finale sur le test

Chaque modèle est réentraîné sur tout le train, puis évalué **une seule fois** sur le test, avec le seuil choisi à l'étape 8.

In [31]:
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

print("Coût si on accepte tout le monde (test) :", f"{y_test.sum() * COUT_DEFAUT_ACCORDE:,.0f} €\n")

for nom, modele in modeles.items():
    modele.fit(X_train, y_train)
    probas_test = modele.predict_proba(X_test)[:, 1]
    refus = probas_test >= seuils_choisis[nom]

    defauts_accordes = ((~refus) & (y_test == 1)).sum()
    bons_refuses = (refus & (y_test == 0)).sum()
    cout = defauts_accordes * COUT_DEFAUT_ACCORDE + bons_refuses * COUT_BON_REFUSE

    print(f"=== {nom} (seuil {seuils_choisis[nom]:.2f}) ===")
    print(f"AUC test : {roc_auc_score(y_test, probas_test):.3f}")
    print(confusion_matrix(y_test, refus.astype(int)))
    print(classification_report(y_test, refus.astype(int), digits=2))
    print(f"Défauts accordés : {defauts_accordes} | bons refusés : {bons_refuses} | coût : {cout:,.0f} €\n")

Coût si on accepte tout le monde (test) : 1,016,000 €



=== Logistique (seuil 0.51) ===
AUC test : 0.796
[[1614  532]
 [  70  184]]
              precision    recall  f1-score   support

           0       0.96      0.75      0.84      2146
           1       0.26      0.72      0.38       254

    accuracy                           0.75      2400
   macro avg       0.61      0.74      0.61      2400
weighted avg       0.88      0.75      0.79      2400

Défauts accordés : 70 | bons refusés : 532 | coût : 599,200 €



=== Forêt (seuil 0.56) ===
AUC test : 0.804
[[1790  356]
 [ 106  148]]
              precision    recall  f1-score   support

           0       0.94      0.83      0.89      2146
           1       0.29      0.58      0.39       254

    accuracy                           0.81      2400
   macro avg       0.62      0.71      0.64      2400
weighted avg       0.88      0.81      0.83      2400

Défauts accordés : 106 | bons refusés : 356 | coût : 637,600 €



=== Boosting réglé (seuil 0.15) ===
AUC test : 0.819
[[1839  307]
 [ 103  151]]
              precision    recall  f1-score   support

           0       0.95      0.86      0.90      2146
           1       0.33      0.59      0.42       254

    accuracy                           0.83      2400
   macro avg       0.64      0.73      0.66      2400
weighted avg       0.88      0.83      0.85      2400

Défauts accordés : 103 | bons refusés : 307 | coût : 596,200 €



| Modèle | AUC validation croisée | AUC test | Coût test |
|---|---|---|---|
| Logistique | 0,791 | 0,796 | 599 200 € |
| Forêt | 0,795 | 0,804 | 637 600 € |
| Boosting réglé | 0,807 | **0,819** | **596 200 €** |

- **La validation croisée était fiable** : l'ordre des AUC est conservé et les écarts restent faibles. Ce découpage train/test est un peu favorable, tous les modèles y gagnent quelques millièmes.
- **En euros, boosting et logistique sont presque à égalité** (3 000 € d'écart). Le test ne contient que 254 défauts : un seul défaut de plus ou de moins vaut 4 000 €, plus que l'écart entre les deux modèles. La forêt passe en dernière position.
- Lecture pour la logistique : précision de 26 % (sur les dossiers refusés, 1 sur 4 aurait vraiment fait défaut, contre 1 sur 10 au hasard) et rappel de 72 % (72 % des défauts sont évités). L'accuracy est à relativiser : un modèle qui ne refuse jamais aurait 89 %.

## 10. Interprétation : qu'est-ce qui pèse dans les décisions ?

Importance par permutation (baisse d'AUC quand on mélange une colonne), calculée de la même façon pour les deux modèles.

In [32]:
from sklearn.inspection import permutation_importance

importances = {}
for nom in ["Boosting réglé", "Logistique"]:
    perm = permutation_importance(
        modeles[nom], X_test, y_test,
        scoring="roc_auc", n_repeats=10, random_state=42, n_jobs=-1
    )
    importances[nom] = pd.Series(perm.importances_mean, index=X_test.columns)

comparaison = pd.DataFrame(importances).sort_values("Boosting réglé", ascending=False)
comparaison[(comparaison > 0.001).any(axis=1)].round(3)

,Boosting réglé,Logistique
Depasse_35,0.174,0.194
Age,0.022,0.011
Nb_Incidents_24M,0.020,0.025
Epargne_EUR,0.019,0.002
Jours_Decouvert_12M,0.010,0.005
Situation_Pro,0.005,0.012
Statut_Logement,0.002,0.002
Taux_Annuel_Pct,0.002,0.001
Taux_Endettement,0.000,0.006
Objet_Pret,0.000,0.001


In [33]:
logit_final = modeles["Logistique"]
noms = logit_final.named_steps["preprocessor"].get_feature_names_out()
coefs = logit_final.named_steps["classifier"].coef_[0]

coefficients = pd.DataFrame({
    "variable": noms,
    "coefficient": coefs,
    "odds_ratio": np.exp(coefs)
}).sort_values("coefficient", key=abs, ascending=False)

coefficients.head(15).round(3)

,variable,coefficient,odds_ratio
14,num__Depasse_35,0.974,2.649
25,cat__Situation_Pro_Fonctionnaire,-0.435,0.647
24,cat__Situation_Pro_CDI,-0.280,0.756
10,num__Nb_Incidents_24M,0.259,1.296
16,num__missingindicator_Anciennete_Emploi_Ans,0.255,1.291
26,cat__Situation_Pro_Indépendant,0.255,1.291
13,num__Taux_Endettement,0.191,1.211
0,num__Age,-0.185,0.831
27,cat__Situation_Pro_Retraité,0.175,1.191
12,num__Mensualite_EUR,-0.168,0.846


- Les deux modèles s'accordent sur le trio de tête : **dépassement des 35 %**, **incidents de paiement**, **âge**.
- `Taux_Endettement` semble inutile, mais c'est un piège de la permutation : quand on le mélange, le modèle s'appuie encore sur `Depasse_35`, qui contient la même information. Les variables redondantes se « cachent » l'une l'autre.
- L'âge pèse deux fois plus dans le boosting, et l'**épargne** presque uniquement dans le boosting : leurs effets ne sont pas des lignes droites.

In [34]:
tranches = pd.cut(credit["Age"], bins=[18, 25, 30, 40, 50, 60, 72, 100])
credit.groupby(tranches, observed=True)["Defaut"].agg(["count", "mean"]).round(3)

,count,mean
Age,,
"(18, 25]",800,0.210
"(25, 30]",933,0.129
"(30, 40]",3282,0.095
"(40, 50]",3386,0.084
"(50, 60]",1666,0.080
"(60, 72]",1147,0.080
"(72, 100]",774,0.203


Le risque a une **forme en U** : environ 21 % de défauts avant 25 ans, 8 % entre 40 et 72 ans, 20 % après 72 ans. La logistique ne peut tracer qu'une droite : elle capte le risque des jeunes mais rate la remontée après 72 ans (son coefficient d'âge est négatif). Le boosting découpe l'âge en tranches et voit les deux extrémités.

Comme pour les 35 %, on peut donner cette forme à la logistique avec une variable métier.

### Logistique enrichie : `Age_Extreme`

In [35]:
# Âge aux deux extrêmes (reste vide si l'âge est inconnu) ; calcul ligne par ligne, sans risque de fuite
for X in (X_train, X_test):
    X["Age_Extreme"] = ((X["Age"] < 25) | (X["Age"] > 72)).astype(float)
    X.loc[X["Age"].isna(), "Age_Extreme"] = np.nan

X_train.assign(Defaut=y_train).groupby("Age_Extreme")["Defaut"].agg(["count", "mean"]).round(3)

,count,mean
Age_Extreme,,
0.0,8452,0.091
1.0,1137,0.213


In [36]:
logit_age = logistique(numeric_features + ["Age_Extreme"])
scores_age = evaluer("Logistique + Age_Extreme", logit_age)
seuil_cout_optimal(cross_val_predict(logit_age, X_train, y_train, cv=cv, method="predict_proba")[:, 1], y_train)

Logistique + Age_Extreme         | AUC apprentissage : 0.809 | AUC validation : 0.800 ± 0.008


seuil                     0.5
refus                  3080.0
defauts_accordes        249.0
bons_refuses           2313.0
cout                2383800.0
Name: 48, dtype: float64

Le test étant déjà ouvert, cette nouvelle logistique est évaluée uniquement par validation croisée. Comme tous les modèles sont notés sur les mêmes plis, on peut les comparer **pli par pli** :

In [37]:
scores_boost = np.array([
    recherche.cv_results_[f"split{i}_test_score"][recherche.best_index_] for i in range(5)
])

pli_par_pli = pd.DataFrame({
    "logistique": scores_logit,
    "logistique_age": scores_age,
    "boosting": scores_boost
})
pli_par_pli["gain_age"] = pli_par_pli["logistique_age"] - pli_par_pli["logistique"]
pli_par_pli["ecart_boosting"] = pli_par_pli["boosting"] - pli_par_pli["logistique_age"]
pli_par_pli.round(3)

,logistique,logistique_age,boosting,gain_age,ecart_boosting
0,0.795,0.801,0.805,0.006,0.004
1,0.788,0.792,0.794,0.005,0.002
2,0.799,0.813,0.812,0.014,-0.001
3,0.781,0.791,0.810,0.010,0.019
4,0.791,0.802,0.812,0.012,0.010


- `Age_Extreme` améliore l'AUC de la logistique sur **les 5 plis** (+0,005 à +0,014) : le gain est fiable, même s'il est plus petit que l'écart-type. AUC moyen : 0,791 → **0,800**.
- Le boosting reste devant sur 4 plis sur 5, avec une égalité : son avance moyenne n'est plus que de 0,007.
- En euros, le gain est faible (−4 200 € sur le train) : les dossiers aux âges extrêmes sont mieux classés, mais ce ne sont pas eux qui font la différence au seuil choisi. L'avance du boosting en euros vient sans doute de l'épargne, que la logistique ne sait pas lire seule.

## Conclusion

| Modèle | AUC validation croisée | AUC test | Coût train (validation croisée) | Coût test |
|---|---|---|---|---|
| Logistique + `Depasse_35` | 0,791 ± 0,006 | 0,796 | 2 388 000 € | 599 200 € |
| Logistique + `Depasse_35` + `Age_Extreme` | 0,800 ± 0,008 | non évalué | 2 383 800 € | non évalué |
| Forêt aléatoire | 0,795 ± 0,007 | 0,804 | 2 365 600 € | 637 600 € |
| **Boosting réglé** | **0,807 ± 0,007** | **0,819** | **2 335 800 €** | **596 200 €** |

### Recommandation
**Je recommande la régression logistique enrichie de variables métier** (`Depasse_35`, `Age_Extreme`).
- **Elle est presque aussi performante que le boosting** : AUC de 0,800 contre 0,807 en validation croisée, soit moins d'un point d'écart.
- **Elle est explicable dossier par dossier** : chaque refus se justifie par des coefficients lisibles (« taux d'endettement au-delà de 35 % », « incidents de paiement récents »). C'est un atout auprès des clients, des conseillers et du régulateur.
- **Elle est plus simple à mettre en production et à maintenir** : peu de paramètres, un comportement prévisible, un suivi facile dans le temps.

### Pourquoi pas le boosting, alors qu'il obtient les meilleurs scores ?
- **Son gain est réel mais modeste.** En validation croisée, il coûte environ 48 000 € de moins que la logistique enrichie sur 9 600 dossiers, soit environ **5 € par dossier**.
- **Ce gain reste incertain.** Sur le test, l'écart en euros entre boosting et logistique n'est que de 3 000 €, moins que le coût d'un seul défaut accordé (4 000 €). Avec 254 défauts dans le test, on ne peut pas affirmer que le boosting ferait gagner de l'argent en production.
- **Il est plus coûteux à expliquer, pas impossible.** Des outils comme SHAP permettent de justifier une décision individuelle, mais ils demandent un travail de mise en place et de validation supplémentaire.
- **Sa marge de progression est faible.** Les 5 meilleurs réglages testés sont tous entre 0,804 et 0,807 : affiner la grille de réglages n'apporterait probablement que quelques millièmes.

### Dans quel cas je basculerais vers le boosting
**Si la banque veut accepter davantage de demandes.** À 31 % de refus, les deux modèles se valent. Quand on réduit les refus (21 %), le boosting laisse passer nettement moins de défauts : il trie mieux la « zone grise » des dossiers sans signal évident. À titre d'ordre de grandeur, et sous ces hypothèses fictives, 5 € par dossier représenteraient environ 500 000 € par an pour 100 000 demandes. Il faudrait comparer ce montant au coût réel de mise en production, de maintenance et d'explication d'un boosting avant de décider.

### Ce que ce projet m'a appris sur le choix d'un modèle
La logistique ne se rapproche du boosting que parce qu'on lui a **décrit les données** : le saut de risque au-delà de 35 % d'endettement, le risque plus élevé aux deux extrémités de l'âge. Ce travail demande du temps, de la réflexion et une bonne compréhension métier. Le boosting, lui, trouve seul ces seuils et ces interactions. Choisir la logistique, c'est donc échanger un peu de performance contre de la transparence, au prix d'un travail d'analyse plus important.

### Ce qui fait le risque de défaut
Les deux modèles s'accordent sur les mêmes facteurs :
- **le dépassement du plafond d'endettement de 35 %**, de loin le plus important (4 % de défauts en dessous, 25 % au-dessus) ;
- **les incidents de paiement** des 24 derniers mois ;
- **l'âge**, avec un risque plus élevé avant 25 ans et après 72 ans (environ 21 % de défauts, contre 8 % entre 40 et 72 ans) ;
- **une épargne faible**, que seul le boosting exploite pleinement.

### Limites et prochaines étapes
- Données entièrement fictives et hypothèses de coûts simplifiées (4 000 € par défaut accordé, 600 € par bon client refusé). Le choix du seuil devrait être refait avec les coûts réels de la banque.
- La logistique enrichie n'a été évaluée que par validation croisée (la variable `Age_Extreme` a été ajoutée après l'ouverture du test). Avant toute mise en production, elle devrait être validée sur des demandes plus récentes (`Date_Demande`).
- Pistes : vérifier que les probabilités sont fiables (calibration), et tester des explications individuelles (SHAP) pour mesurer le coût réel d'un boosting explicable.